In [1]:
# Standard library
import gc
import warnings
from typing import Union, Optional, Tuple, List, Dict, Any
from pathlib import Path
import pickle
import base64

# Core data science
import numpy as np
import pandas as pd

# Scikit-learn
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_squared_error

from PIL import Image
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

DATA_DIR = Path("/share/crsp/lab/pkaiser/ddlin/single-cell-multimodal-ml/data")
RAW_DIR = DATA_DIR.joinpath("raw")
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

FP_CITE_TRAIN_INPUTS  = RAW_DIR.joinpath("train_cite_inputs.h5")
FP_CITE_TRAIN_TARGETS = RAW_DIR.joinpath("train_cite_targets.h5")
FP_CITE_TEST_INPUTS   = RAW_DIR.joinpath("test_cite_inputs.h5")

FP_IMPORTANT_COLS = PROCESSED_DIR.joinpath("important_cols.txt")
FP_CONSTANT_COLS = PROCESSED_DIR.joinpath("constant_cols.txt")


FP_EVALUATION_IDS     = RAW_DIR.joinpath("evaluation_ids.csv")
FP_CELL_METADATA      = RAW_DIR.joinpath("metadata.csv")

VERBOSE = 0

## Test predictions for Multiome

In [ ]:
# Load the test sparce matrix
TEST_INPUT_NPZ = PROCESSED_DIR.joinpath("test_multi_inputs_values.npz")
multi_test_x = scipy.sparse.load_npz(TEST_INPUT_NPZ)

# Apply training PCA to get it ready for prediction
multi_test_x = svd_inputs.transform(multi_test_x)
# multi_test_x = multi_test_x[:,:40]
multi_test_x.shape

(55935, 512)

In [ ]:
eval_ids = pd.read_parquet(DATA_DIR.joinpath("processed", "evaluation.parquet"))
eval_ids.cell_id = eval_ids.cell_id.astype(pd.CategoricalDtype())
eval_ids.gene_id = eval_ids.gene_id.astype(pd.CategoricalDtype())

submission = pd.Series(name='target',
                       index=pd.MultiIndex.from_frame(eval_ids), 
                       dtype=np.float32)
submission

row_id    cell_id       gene_id        
0         c2150f55becb  CD86              NaN
1         c2150f55becb  CD274             NaN
2         c2150f55becb  CD270             NaN
3         c2150f55becb  CD155             NaN
4         c2150f55becb  CD112             NaN
                                           ..
65744175  2c53aa67933d  ENSG00000134419   NaN
65744176  2c53aa67933d  ENSG00000186862   NaN
65744177  2c53aa67933d  ENSG00000170959   NaN
65744178  2c53aa67933d  ENSG00000107874   NaN
65744179  2c53aa67933d  ENSG00000166012   NaN
Name: target, Length: 65744180, dtype: float32

In [ ]:
TEST_META = np.load(
    PROCESSED_DIR.joinpath("test_multi_inputs_metadata.npz"),
    allow_pickle=True
    )
TEST_META

NpzFile '/share/crsp/lab/pkaiser/ddlin/single-cell-multimodal-ml/data/processed/test_multi_inputs_metadata.npz' with keys: index, columns, shape, nnz, sparsity...

In [ ]:
TEST_META = np.load(
    PROCESSED_DIR.joinpath("test_multi_inputs_metadata.npz"),
    allow_pickle=True
    )


y_columns = TEST_META["columns"]
test_index = TEST_META["index"]

cell_dict = dict((k,v) for v,k in enumerate(test_index)) 
assert len(cell_dict)  == len(test_index)

gene_dict = dict((k,v) for v,k in enumerate(y_columns))
assert len(gene_dict) == len(y_columns)

eval_ids_cell_num = eval_ids.cell_id.apply(lambda x:cell_dict.get(x, -1))
eval_ids_gene_num = eval_ids.gene_id.apply(lambda x:gene_dict.get(x, -1))
valid_multi_rows = (eval_ids_gene_num !=-1) & (eval_ids_cell_num!=-1)

submission.iloc[valid_multi_rows] = preds[eval_ids_cell_num[valid_multi_rows].to_numpy(),
eval_ids_gene_num[valid_multi_rows].to_numpy()]

del eval_ids_cell_num, eval_ids_gene_num, valid_multi_rows, eval_ids, test_index, y_columns
gc.collect()

submission

NameError: name 'eval_ids' is not defined

In [ ]:
# This is the test label dense matrix shape
preds = np.zeros((multi_test_x.shape[0], 23418), dtype='float16')

for fold in range(N_SPLITS):
    print(f'fold {fold} prediction')
    model_path = model_dir / f"model_{fold}.keras"
    model = tf.keras.models.load_model(model_path)

    # From the predicted PCs, reconstruct the dense matrix
    preds += (model.predict(multi_test_x)@svd_targets.components_)/N_SPLITS
    # model.predict(multi_test_x)

    gc.collect()

fold 0 prediction
1748/1748 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step
fold 1 prediction
1748/1748 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step
fold 2 prediction
1748/1748 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step


## Total submission

In [ ]:
submission.reset_index(drop=True, inplace=True)
submission.index.name = 'row_id'

cite_submission = pd.read_csv("submission_lolo_1.csv")
cite_submission = cite_submission.set_index("row_id")
cite_submission = cite_submission["target"]
submission[submission.isnull()] = cite_submission[submission.isnull()]
submission
# == > score 0.812


row_id
0           0.094605
1          -0.162362
2          -0.405332
3          -0.302582
4           1.114355
              ...   
65744175    2.785156
65744176   -0.379150
65744177   -0.375732
65744178    0.167603
65744179    2.480469
Name: target, Length: 65744180, dtype: float32

In [ ]:
sub_ensembling = pd.read_csv('../input/5-5-msci22-ensembling-citeseq/submission.csv')
submission1 = sub_ensembling.copy()
submission1['target'] = 0.4 * submission + 0.6 * sub_ensembling['target']
submission1

,row_id,target
0,0,0.182096
1,1,0.010811
2,2,-0.101281
3,3,1.442052
4,4,2.616124
...,...,...
65744175,65744175,5.422715
65744176,65744176,-0.302524
65744177,65744177,-0.293852
65744178,65744178,0.755234


In [ ]:
submission1.to_csv("submission_lolo_total_ensembling.csv", index = False)